# Information Density in Active Learning - Starter Notebook
This notebook combines uncertainty with density weighting so selected points are informative and representative of the data pool.

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

In [ ]:
X, y = make_classification(n_samples=1200, n_features=9, n_informative=6, random_state=42)
X_labeled, y_labeled = X[:150], y[:150]
X_pool = X[150:700]

model = LogisticRegression(max_iter=1000)
model.fit(X_labeled, y_labeled)
probs = model.predict_proba(X_pool)

uncertainty = -np.sum(probs * np.log(probs + 1e-12), axis=1)
dist_matrix = cdist(X_pool, X_pool, metric='euclidean')
density = np.exp(-dist_matrix).mean(axis=1)
informative_density_score = uncertainty * density
query_ids = np.argsort(informative_density_score)[-25:]

In [ ]:
pd.DataFrame({
    'pool_index': query_ids,
    'uncertainty': uncertainty[query_ids],
    'density': density[query_ids],
    'information_density_score': informative_density_score[query_ids],
}).sort_values('information_density_score', ascending=False).head(10).round(5)